# Пример - Метрические параметры графов

## Подготовка

Выполните все ячейки сверху вниз. Ревизия пакета: **14f137c**.

In [1]:
from importlib.metadata import PackageNotFoundError, distribution
import json
import subprocess
import sys

source = "https://github.com/magin67/polyform_algebra/archive/14f137c0b4bf965392f47f877165f2f3b8168073.zip"
try:
    installed = distribution("polyform_algebra")
    installed_url = json.loads(installed.read_text("direct_url.json") or "{}").get("url")
except PackageNotFoundError:
    installed = None
    installed_url = None

# A tag and a development revision can report the same version string.
if installed_url != source:
    if "polyform_algebra" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned package revision.")
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if installed is not None and installed.version == "0.9.0":
        command += ["--force-reinstall", "--no-deps"]
    subprocess.check_call(command + [source])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


Восемь расчётов сопровождают восемь заметок серии «Метрические параметры графов». Используем рациональные дроби для метрических и остовных параметров и численный метод для спектральной центральности.


Выполняйте фрагменты сверху вниз. Блокнот устанавливает проверенную ревизию пакета, в которой доступны точные операции серии. [polyform-algebra 14f137c](https://github.com/magin67/polyform_algebra/commit/14f137c0b4bf965392f47f877165f2f3b8168073).


## Радиус и диаметр пути и звезды

Сравним путь $a-b-c-d$ и звезду с центром $a$. Все проводимости единичны. Точный расчёт даёт $\rho=3/4$ и $D_{\mathrm{res}}=4\rho=3$ для обоих деревьев. Максимальный попарный резистенс равен $3$ на пути и $2$ на звезде. Би-координаты центров различаются, а координата центра сферы при центральной вершине звезды отрицательна.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D1%8B%D0%B9-%D1%80%D0%B0%D0%B4%D0%B8%D1%83%D1%81-%D0%B8-%D0%B4%D0%B8%D0%B0%D0%BC%D0%B5%D1%82%D1%80-%D0%B3%D1%80%D0%B0%D1%84%D0%B0#%D0%BF%D1%80%D0%BE%D1%81%D1%82%D1%8B%D0%B5-%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D1%8B).

In [3]:
from fractions import Fraction as Q
from itertools import combinations
from math import isclose

from networkx import Graph
from polyform_algebra.core import (
    Network, circumsphere, degree_potential, evaluate_form, green, induced_subgraph,
    kron_reduce, laplacian, laplacian_circumsphere, laplacian_form, network,
    resistance, simplex_volume_squared, spanning_tree_weight,
)
from polyform_algebra.matrix.networkx import from_networkx

vertices = ("a", "b", "c", "d")
path = network(vertices, (("a", "b", 1), ("b", "c", 1), ("c", "d", 1)))
star = network(vertices, (("a", "b", 1), ("a", "c", 1), ("a", "d", 1)))
graphs = {"P4": path, "K1,3": star}

spheres = {name: circumsphere(graph).value for name, graph in graphs.items()}
assert spheres["P4"].center_coordinates.values == (Q(1, 2), 0, 0, Q(1, 2))
assert spheres["K1,3"].center_coordinates.values == (Q(-1, 2), Q(1, 2), Q(1, 2), Q(1, 2))
maximum_resistances = {}
for name, graph in graphs.items():
    sphere = spheres[name]
    rho = sphere.resistance_radius
    center = ", ".join(
        f"{vertex}={value}" for vertex, value in zip(vertices, sphere.center_coordinates.values)
    )
    maximum_resistances[name] = max(max(row) for row in resistance(graph).value.values)
    assert rho == Q(3, 4)
    print(f"{name}: rho={rho}, D_res={4 * rho}, max R={maximum_resistances[name]}")
    print(f"  center: {center}")
assert maximum_resistances == {"P4": 3, "K1,3": 2}

P4: rho=3/4, D_res=3, max R=3
  center: a=1/2, b=0, c=0, d=1/2
K1,3: rho=3/4, D_res=3, max R=2
  center: a=-1/2, b=1/2, c=1/2, d=1/2


## Радиус и объём поднабора

Возьмём единичный полный граф $K_4$. Его резистивный симплекс - правильный тетраэдр. Чтобы вычислить радиус грани $abc$ в метрике всего графа, исключим вершину $d$ по Крону. Для квадрата площади используем исходный граф и выбранные вершины. Для тетраэдра получаем $\rho=3/16$ и $V^2=1/576$, для грани $\rho=1/6$ и $S^2=3/64$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D1%8B%D0%B9-%D1%80%D0%B0%D0%B4%D0%B8%D1%83%D1%81-%D0%BD%D0%B0%D0%B1%D0%BE%D1%80%D0%B0---%D0%B3%D1%80%D0%B0%D0%BC%D0%B8%D0%B0%D0%BD-%D0%B8-%D0%BE%D0%B1%D1%8A%D1%91%D0%BC#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D1%8B-%D0%B8-%D0%BF%D1%80%D0%BE%D0%B2%D0%B5%D1%80%D0%BA%D0%B0-%D0%BD%D0%BE%D1%80%D0%BC%D0%B8%D1%80%D0%BE%D0%B2%D0%BE%D0%BA).

In [4]:
tetrahedron = network(vertices, tuple((i, j, 1) for i, j in combinations(vertices, 2)))
face = ("a", "b", "c")
face_network = kron_reduce(tetrahedron, face).value.network
tetrahedron_radius = circumsphere(tetrahedron).value.resistance_radius
face_radius = circumsphere(face_network).value.resistance_radius
volume_squared = simplex_volume_squared(tetrahedron, vertices).value
face_area_squared = simplex_volume_squared(tetrahedron, face).value
assert (tetrahedron_radius, volume_squared) == (Q(3, 16), Q(1, 576))
assert (face_radius, face_area_squared) == (Q(1, 6), Q(3, 64))
print(f"abcd: rho={tetrahedron_radius}, V²={volume_squared}")
print(f"abc: rho={face_radius}, area²={face_area_squared}")

abcd: rho=3/16, V²=1/576
abc: rho=1/6, area²=3/64


## Три центральности

На том же пути и звезде сравним центроидную, остовную и спектральную центральности. Центроидные оценки пропорциональны обратным диагональным элементам матрицы Грина. Остовные оценки пропорциональны средним остовным степеням $\delta_i=2(1-\alpha^i)$. Обе колонки вычисляются точно. Для спектральной колонки используем численный собственный вектор матрицы смежности. Каждая колонка нормирована к единичной сумме.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B/%D1%86%D0%B5%D0%BD%D1%82%D1%80%D0%BE%D0%B8%D0%B4%D0%BD%D0%B0%D1%8F,-%D0%BE%D1%81%D1%82%D0%BE%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D0%B0%D1%8F-%D1%86%D0%B5%D0%BD%D1%82%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D0%B8#%D0%BF%D0%BE%D0%BB%D0%BD%D1%8B%D0%B9-%D0%B3%D1%80%D0%B0%D1%84-%D0%BF%D1%83%D1%82%D1%8C-%D0%B8-%D0%B7%D0%B2%D0%B5%D0%B7%D0%B4%D0%B0).

In [5]:
def centralities(graph, sphere):
    Gr = green(graph).value.values
    inverse_norms = tuple(1 / row[i] for i, row in enumerate(Gr))
    centroid = tuple(value / sum(inverse_norms) for value in inverse_norms)
    expected_degrees = tuple(2 * (1 - x) for x in sphere.center_coordinates.values)
    tree_scores = tuple(value / sum(expected_degrees) for value in expected_degrees)

    nx_graph = Graph()
    nx_graph.add_nodes_from(graph.context.labels)
    # Convert conductances at the boundary of the numerical spectral API.
    nx_graph.add_weighted_edges_from((a, b, float(c)) for a, b, c in graph.edges)
    matrix = from_networkx(nx_graph, vertices=graph.context.labels)
    numeric_tree = matrix.spanning_tree_centrality().scores.to_numpy()
    eigenvector = matrix.eigenvector_centrality().scores.to_numpy()
    assert all(isclose(float(x), y, abs_tol=1e-12) for x, y in zip(tree_scores, numeric_tree))
    assert sum(centroid) == sum(tree_scores) == 1
    assert isclose(sum(eigenvector), 1)
    return centroid, tree_scores, eigenvector

for name, graph in graphs.items():
    centroid, tree_scores, eigenvector = centralities(graph, spheres[name])
    if name == "P4":
        assert centroid == (Q(3, 20), Q(7, 20), Q(7, 20), Q(3, 20))
        assert tree_scores == (Q(1, 6), Q(1, 3), Q(1, 3), Q(1, 6))
    else:
        assert centroid == (Q(11, 20), Q(3, 20), Q(3, 20), Q(3, 20))
        assert tree_scores == (Q(1, 2), Q(1, 6), Q(1, 6), Q(1, 6))
    print(name, "  vertex   centroid   tree     eigenvector")
    for label, c, t, e in zip(vertices, centroid, tree_scores, eigenvector):
        print(f"     {label:>5} {str(c):>10} {str(t):>7} {e:>14.6f}")

P4   vertex   centroid   tree     eigenvector
         a       3/20     1/6       0.190983
         b       7/20     1/3       0.309017
         c       7/20     1/3       0.309017
         d       3/20     1/6       0.190983
K1,3   vertex   centroid   tree     eigenvector
         a      11/20     1/2       0.366025
         b       3/20     1/6       0.211325
         c       3/20     1/6       0.211325
         d       3/20     1/6       0.211325


## Две меры компактности

Путь и звезда имеют одинаковое число вершин, суммарную проводимость и резистивный радиус. Радиусная компактность $\eta_\rho$ одинакова и равна $1/2$. Компактность $\eta_K$ учитывает все попарные резистенсы и даёт $3/5$ на пути и $2/3$ на звезде. Проверим, что умножение всех проводимостей на $3$ сохраняет оба безразмерных показателя.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B/%D0%BF%D0%BB%D0%BE%D1%82%D0%BD%D0%BE%D1%81%D1%82%D1%8C,-%D0%BA%D0%BE%D0%BC%D0%BF%D0%B0%D0%BA%D1%82%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D0%B8-%D0%B4%D1%80%D0%B5%D0%B2%D0%B5%D1%81%D0%BD%D0%B0%D1%8F-%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D0%BD%D0%BE%D1%81%D1%82%D1%8C#%D0%B4%D1%80%D0%B5%D0%B2%D0%B5%D1%81%D0%BD%D0%B0%D1%8F-%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D0%BD%D0%BE%D1%81%D1%82%D1%8C).

In [6]:
def compactness(graph, rho):
    R = resistance(graph).value.values
    n = len(graph.context.labels)
    mean_degree = 2 * sum(c for _, _, c in graph.edges) / n
    kirchhoff = sum(R[i][j] for i, j in combinations(range(n), 2))
    radial = Q((n - 1)**2, n**2) / (mean_degree * rho)
    by_kirchhoff = (n - 1)**2 / (mean_degree * kirchhoff)
    return radial, by_kirchhoff

assert compactness(path, spheres["P4"].resistance_radius) == (Q(1, 2), Q(3, 5))
assert compactness(star, spheres["K1,3"].resistance_radius) == (Q(1, 2), Q(2, 3))
for name, graph in graphs.items():
    radial, by_kirchhoff = compactness(graph, spheres[name].resistance_radius)
    scaled = network(vertices, tuple((a, b, 3 * c) for a, b, c in graph.edges))
    assert compactness(scaled, circumsphere(scaled).value.resistance_radius) == (radial, by_kirchhoff)
    print(f"{name}: eta_rho={radial}, eta_K={by_kirchhoff}")

P4: eta_rho=1/2, eta_K=3/5
K1,3: eta_rho=1/2, eta_K=2/3


## Два способа удалить вершину

В единичном треугольнике оставим вершины $a,b$. При удалении $c$ вместе со связями остаётся исходное ребро проводимости $1$. При исключении по Крону его проводимость становится $3/2$, чтобы сохранить исходный резистенс. Сравним радиусы и остовные коэффициенты.

| Граф | $\tau$ | $\rho$ |
| --- | ---: | ---: |
| Исходный треугольник | $3$ | $2/9$ |
| После удаления | $1$ | $1/4$ |
| После исключения | $3/2$ | $1/6$ |

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B/%D1%83%D0%B4%D0%B0%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD---%D0%BE%D1%81%D1%82%D0%BE%D0%B2%D0%BD%D0%BE%D0%B9-%D0%BA%D0%BE%D1%8D%D1%84%D1%84%D0%B8%D1%86%D0%B8%D0%B5%D0%BD%D1%82-%D0%B8-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D1%8B%D0%B9-%D1%80%D0%B0%D0%B4%D0%B8%D1%83%D1%81#%D0%BC%D0%B0%D0%BB%D1%8B%D0%B5-%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D1%8B).

In [7]:
triangle = network(face, tuple((i, j, 1) for i, j in combinations(face, 2)))
remaining = ("a", "b")
deleted = induced_subgraph(triangle, remaining).value
eliminated = kron_reduce(triangle, remaining).value.network
variants = (("K3", triangle), ("Deletion", deleted), ("Elimination", eliminated))
tree_weights = tuple(spanning_tree_weight(graph).value for _, graph in variants)
radii = tuple(circumsphere(graph).value.resistance_radius for _, graph in variants)
assert tree_weights == (3, 1, Q(3, 2))
assert radii == (Q(2, 9), Q(1, 4), Q(1, 6))
assert tuple(value / tree_weights[0] for value in tree_weights[1:]) == (Q(1, 3), Q(1, 2))
assert tuple(value / radii[0] for value in radii[1:]) == (Q(9, 8), Q(3, 4))
for (label, _), tau, rho in zip(variants, tree_weights, radii):
    print(f"{label}: tau={tau}, rho={rho}")

K3: tau=3, rho=2/9
Deletion: tau=1, rho=1/4
Elimination: tau=3/2, rho=1/6


## Центры двух сфер

Зададим проводимости $c_{ab}=7$, $c_{ac}=c_{bc}=1$. Сравним центр с би-координатами $\alpha$ в резистивной геометрии, где грамиан равен $\mathbf{Gr}$, и центр $\beta$ в лапласиановой геометрии, где грамиан равен $\mathbf L$. Для $\beta$ координата при $c$ отрицательна. Здесь $\rho$ и $\sigma$ обозначают квадраты линейных радиусов в двух разных геометриях.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B/%D1%86%D0%B5%D0%BD%D1%82%D1%80%D1%8B-%D1%81%D1%84%D0%B5%D1%80-%D0%B8-%D0%BD%D0%B5%D0%BE%D0%B4%D0%BD%D0%BE%D1%80%D0%BE%D0%B4%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D0%B3%D1%80%D0%B0%D1%84%D0%B0#%D1%82%D0%BE%D1%87%D0%BD%D1%8B%D0%B5-%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D1%8B-%D0%B8-%D0%BE%D1%82%D1%80%D0%B8%D1%86%D0%B0%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D1%8B%D0%B5-%D0%BA%D0%BE%D0%BE%D1%80%D0%B4%D0%B8%D0%BD%D0%B0%D1%82%D1%8B).

In [8]:
weighted_triangle = network(face, (("a", "b", 7), ("a", "c", 1), ("b", "c", 1)))
resistance_sphere = circumsphere(weighted_triangle).value
laplacian_sphere = laplacian_circumsphere(weighted_triangle).value
alpha = resistance_sphere.center_coordinates.values
beta = laplacian_sphere.center_coordinates.values
assert alpha == (Q(4, 15), Q(4, 15), Q(7, 15))
assert resistance_sphere.resistance_radius == Q(32, 225)
assert beta == (Q(2, 3), Q(2, 3), Q(-1, 3))
assert laplacian_sphere.squared_radius == 8
print("alpha:", ", ".join(map(str, alpha)), "| rho =", resistance_sphere.resistance_radius)
print("beta: ", ", ".join(map(str, beta)), "| sigma =", laplacian_sphere.squared_radius)

alpha: 4/15, 4/15, 7/15 | rho = 32/225
beta:  2/3, 2/3, -1/3 | sigma = 8


## Потенциал степеней на пути

Вернёмся к единичному пути $a-b-c$. Степени вершин равны $(1,2,1)$, а отклонения от средней степени равны $(-1,2,-1)/3$. Проверим, что потенциал $\varphi$ имеет нулевую сумму и удовлетворяет уравнению $\mathbf L\varphi=d-\bar d\mathbf1$. Середина пути имеет наибольший потенциал. Координаты $\beta$ получаются сдвигом равномерных координат на $\varphi/2$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B/%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB-%D1%81%D1%82%D0%B5%D0%BF%D0%B5%D0%BD%D0%B5%D0%B9-%D0%B8-%D1%86%D0%B5%D0%BD%D1%82%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D0%B8#%D0%BF%D0%B5%D1%80%D0%B2%D1%8B%D0%B9-%D1%80%D0%B0%D1%81%D1%87%D1%91%D1%82-%D0%BF%D1%83%D1%82%D1%8C-%D0%B8%D0%B7-%D1%82%D1%80%D1%91%D1%85-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD).

In [9]:
short_path = network(face, (("a", "b", 1), ("b", "c", 1)))
phi = degree_potential(short_path).value.values
L = laplacian(short_path).value.values
degrees = tuple(row[i] for i, row in enumerate(L))
mean_degree = sum(degrees) / len(degrees)
source = tuple(value - mean_degree for value in degrees)
beta = laplacian_circumsphere(short_path).value.center_coordinates.values
assert sum(phi) == 0 and phi == (Q(-1, 9), Q(2, 9), Q(-1, 9))
assert tuple(sum(x * y for x, y in zip(row, phi)) for row in L) == source
assert beta == (Q(5, 18), Q(4, 9), Q(5, 18))
print("phi:", ", ".join(map(str, phi)))
print("beta:", ", ".join(map(str, beta)))

phi: -1/9, 2/9, -1/9
beta: 5/18, 4/9, 5/18


## Норма внутреннего лапласиана

В единичном цикле $C_4$ сравним пару соседних вершин, пару противоположных и тройку. Для каждой группы строим форму только из её внутренних связей в исходном контексте, затем оцениваем её в метрике всего цикла. Сверим результат с суммой $c_{ij}R_{ij}$ по внутренним рёбрам. Для тройки получаем $3/2$ в метрике цикла и $2$ при отдельном расчёте в её индуцированном графе.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B/%D0%BD%D0%BE%D1%80%D0%BC%D1%8B-%D0%BB%D0%B0%D0%BF%D0%BB%D0%B0%D1%81%D0%B8%D0%B0%D0%BD%D0%BE%D0%B2-%D0%BF%D0%BE%D0%B4%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2#%D1%82%D1%80%D0%B5%D1%83%D0%B3%D0%BE%D0%BB%D1%8C%D0%BD%D0%B8%D0%BA-%D0%B8-%D0%BA%D0%B2%D0%B0%D0%B4%D1%80%D0%B0%D1%82).

In [10]:
cycle = network(vertices, (("a", "b", 1), ("b", "c", 1), ("c", "d", 1), ("d", "a", 1)))

def internal_norm(graph, labels):
    # Keep the original vertex context so the form is evaluated in G.
    selected = set(labels)
    edges = tuple(edge for edge in graph.edges if edge[0] in selected and edge[1] in selected)
    internal = Network(graph.context, edges, graph.policy)
    value = evaluate_form(graph, laplacian_form(internal).value).value
    pairs = tuple((i, j) for i, j, _ in edges)
    pair_resistances = resistance(graph, pairs=pairs).value.values
    assert value == sum(edge[2] * r for edge, r in zip(edges, pair_resistances))
    return value

for selected, expected in (
    (("a", "b"), Q(3, 4)), (("a", "c"), 0), (face, Q(3, 2)), (vertices, 3),
):
    value = internal_norm(cycle, selected)
    assert value == expected
    print("".join(selected), ":", value)

induced = induced_subgraph(cycle, face).value
assert internal_norm(induced, face) == 2
print("abc evaluated in its own induced graph:", internal_norm(induced, face))

ab : 3/4
ac : 0
abc : 3/2
abcd : 3
abc evaluated in its own induced graph: 2
